In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [3]:
import os
import sys

import io
from io import BytesIO
import csv

import google.auth
from google.cloud import bigquery
#from google.cloud import bigquery_storage

In [6]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "./market-analysis-project-91130-9f9a036682b6.json"

credentials, project_id = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

bqclient = bigquery.Client(credentials=credentials, project=project_id,)

In [8]:

sql = f"""
select * from stck.atlas_sales_all 
where Category='Mattresses & Box Springs' and (WeekId between 202514 and 202526)
"""

df = bqclient.query(sql).to_dataframe()

In [10]:
df

,RetailerId,RetailerName,RetailerSku,upc,ModelNumber,Title,Brand,Category,SubCategory,WeekId,WeekEnding,RetailSales,UnitsSold,RetailPrice
0,1,Amazon.com,B0D4Z5STNM,NaN,LTBS10,10 Inch California King Box Spring High Profil...,Lutown-Teen,Mattresses & Box Springs,Box Springs,202516,2025-04-19,875.94,6,145.99
1,1,Amazon.com,B0D4Z5LQXF,NaN,LTBS10,"10 Inch Full Size Box Spring High Profile, Hea...",Lutown-Teen,Mattresses & Box Springs,Box Springs,202516,2025-04-19,1099.90,10,109.99
2,1,Amazon.com,B0D4Z4VZVC,NaN,LTBS10,"10 Inch Queen Box Spring High Profile, Heavy D...",Lutown-Teen,Mattresses & Box Springs,Box Springs,202516,2025-04-19,959.92,8,119.99
3,1,Amazon.com,B0D4Z4D4H7,NaN,LTBS10,"10 Inch Twin Box Spring High Profile, Heavy Du...",Lutown-Teen,Mattresses & Box Springs,Box Springs,202516,2025-04-19,77.99,1,77.99
4,1,Amazon.com,B0D3XGHVF7,NaN,None,"10 Inch Twin Box Spring and Fabric Cover Set, ...",Artimorany,Mattresses & Box Springs,Box Springs,202516,2025-04-19,479.92,8,59.99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86839,1,Amazon.com,B06XW7H1R4,8.898601e+11,07FM01F,"Olee Sleep Full Mattress, 7 Inch Deluxe Gel Me...",Olee Sleep,Mattresses & Box Springs,Mattresses,202522,2025-05-31,178.99,1,178.99
86840,1,Amazon.com,B07CYJH1LL,8.112010e+11,DFADH1050SH,"Arctic Dreams 10"" Hybrid Cooling Gel Mattress ...",Dreamfoam Bedding,Mattresses & Box Springs,Mattresses,202522,2025-05-31,2237.20,4,559.30
86841,1,Amazon.com,B07VGG1G42,NaN,medium firm mattress,"Vesgantti Full Size Mattress, 10 Inch Hybrid F...",Vesgantti,Mattresses & Box Springs,Mattresses,202522,2025-05-31,5889.69,31,189.99
86842,1,Amazon.com,B01MTEDHQJ,NaN,Chiro Premier Orthopedic,Dream Solutions USA Chiro Premier Orthopedic (...,Dream Solutions USA,Mattresses & Box Springs,Mattress & Box Spring Sets,202522,2025-05-31,734.34,2,367.17


In [12]:
df.to_csv("stckln_mattress_2025_2q_data.csv", index=False)

### 데이터 탐색

In [77]:
df = pd.read_csv('amz_pdp_price_sales_gv_0618.csv')

In [78]:
df['crawl_date'] = pd.to_datetime(df['crawl_date'])
print(df['crawl_date'].min())
print(df['crawl_date'].max())
print(df['crawl_date'].max() - df['crawl_date'].min())

2023-08-17 00:00:00
2025-06-16 00:00:00
669 days 00:00:00


In [79]:
df_2025 = df[df['crawl_date'].dt.year == 2025]
print(df_2025)

        crawl_date        asin  rating  ratings_total  salesrank1  salesrank2  \
2008339 2025-01-01  B00FJNQCEQ     4.3        10462.0     42043.0       127.0   
2008340 2025-01-01  B07KD6VRK2     4.1        13608.0     22304.0        14.0   
2008341 2025-01-01  B07KK8CRT8     4.5          679.0   1168677.0       165.0   
2008342 2025-01-01  B09SFTPS2T     1.0            1.0    875468.0      3707.0   
2008343 2025-01-01  B08G3N4M58     NaN            NaN    958226.0        81.0   
...            ...         ...     ...            ...         ...         ...   
2636767 2025-06-16  B08DXXF9Y4     4.1         1566.0         NaN         NaN   
2636768 2025-06-16  B097QVK388     4.3        10513.0         NaN         NaN   
2636769 2025-06-16  B06WW43FVV     4.6        17182.0         NaN         NaN   
2636770 2025-06-16  B097QXKWMP     4.3        22297.0         NaN         NaN   
2636771 2025-06-16  B0D17M8677     4.6        38023.0         NaN         NaN   

         salesrank3  bw_pri

In [83]:
df_2025.groupby('crawl_date')['asin'].nunique().reset_index().sort_values('crawl_date', ascending=False)

,crawl_date,asin
166,2025-06-16,1478
165,2025-06-15,1479
164,2025-06-14,1482
163,2025-06-13,2084
162,2025-06-12,2140
...,...,...
4,2025-01-05,3987
3,2025-01-04,4024
2,2025-01-03,3986
1,2025-01-02,3976


In [85]:
# asin별로 revenue 합계 집계 (결측치는 0으로 대체)
asin_sales = df_2025.groupby('asin')['revenue'].sum().fillna(0).reset_index()
asin_sales = asin_sales.sort_values('revenue', ascending=False).reset_index(drop=True)
print(asin_sales)

            asin       revenue
0     B0CKYZ3B83  8.210468e+06
1     B0CKZ1CK1H  6.284006e+06
2     B0CKYZC93L  4.709192e+06
3     B0CKYYHD47  4.429654e+06
4     B0CKYZCVXK  3.412454e+06
...          ...           ...
7984  B0F8483X4W  0.000000e+00
7985  B077XJBJ5K -1.421085e-14
7986  B06XJ64MLC -1.900000e+01
7987  B07C3KBRLF -4.399000e+01
7988  B0134FXO4G -6.240000e+02

[7989 rows x 2 columns]


In [87]:
# 4. 전체 매출 합계 및 누적합, 누적비율 계산
total_revenue = asin_sales['revenue'].sum()
asin_sales['cumsum'] = asin_sales['revenue'].cumsum()
asin_sales['cumrate'] = asin_sales['cumsum'] / total_revenue
print(asin_sales)

            asin       revenue        cumsum   cumrate
0     B0CKYZ3B83  8.210468e+06  8.210468e+06  0.059899
1     B0CKZ1CK1H  6.284006e+06  1.449447e+07  0.105744
2     B0CKYZC93L  4.709192e+06  1.920367e+07  0.140100
3     B0CKYYHD47  4.429654e+06  2.363332e+07  0.172416
4     B0CKYZCVXK  3.412454e+06  2.704577e+07  0.197312
...          ...           ...           ...       ...
7984  B0F8483X4W  0.000000e+00  1.370718e+08  1.000005
7985  B077XJBJ5K -1.421085e-14  1.370718e+08  1.000005
7986  B06XJ64MLC -1.900000e+01  1.370718e+08  1.000005
7987  B07C3KBRLF -4.399000e+01  1.370718e+08  1.000005
7988  B0134FXO4G -6.240000e+02  1.370711e+08  1.000000

[7989 rows x 4 columns]


In [89]:
# 5. 상위 80%를 차지하는 asin 리스트 추출
asin_top80 = asin_sales[asin_sales['cumrate'] <= 0.8]['asin'].tolist()
print(len(asin_top80))

113


In [91]:
df_top80 = df[df['asin'].isin(asin_top80)]
print(df_top80)

        crawl_date        asin  rating  ratings_total  salesrank1  salesrank2  \
51      2023-08-17  B09XM73Z65     4.5        78306.0      1009.0         2.0   
93      2023-08-17  B095W72PK1     4.5          253.0     52971.0        67.0   
144     2023-08-17  B075GW4GXH     4.6        19051.0      2932.0         3.0   
233     2023-08-17  B07DGHWVN8     4.6         2888.0      8718.0         1.0   
304     2023-08-17  B072HTRVM1     4.5        13738.0     26994.0       152.0   
...            ...         ...     ...            ...         ...         ...   
2636720 2025-06-16  B0CSJX57TM     4.4         5208.0         NaN         NaN   
2636735 2025-06-16  B0CKYYF9SJ     4.2        16430.0         NaN         NaN   
2636741 2025-06-16  B0CKYYHRN6     4.4        78245.0         NaN         NaN   
2636755 2025-06-16  B0CKYSMM1J     4.3        30283.0         NaN         NaN   
2636769 2025-06-16  B06WW43FVV     4.6        17182.0         NaN         NaN   

         salesrank3  bw_pri

In [93]:
df_top80['asin'].nunique()

113

In [109]:
# 해당 컬럼들 중 하나라도 NaN이면 제거
df_cleaned = df_top80.dropna(subset=['rating', 'ratings_total', 'bw_price', 'units', 'gv']).copy()
#df_cleaned = df_top80.rename(columns={'ratings_total': 'ratings_cnt'})
#df_cleaned = df_cleaned.rename(columns={'ratings_total': 'ratings_cnt'}, inplace=True)
print(df_cleaned)

        crawl_date        asin  rating  ratings_total  salesrank1  salesrank2  \
51      2023-08-17  B09XM73Z65     4.5        78306.0      1009.0         2.0   
93      2023-08-17  B095W72PK1     4.5          253.0     52971.0        67.0   
144     2023-08-17  B075GW4GXH     4.6        19051.0      2932.0         3.0   
233     2023-08-17  B07DGHWVN8     4.6         2888.0      8718.0         1.0   
304     2023-08-17  B072HTRVM1     4.5        13738.0     26994.0       152.0   
...            ...         ...     ...            ...         ...         ...   
2635790 2025-06-13  B0CKYYMG6F     4.4        78216.0       399.0         4.0   
2635807 2025-06-13  B0CSJTBM1L     4.4         9849.0       693.0         7.0   
2635842 2025-06-13  B0CSJX57TM     4.4         5200.0      3714.0        18.0   
2635853 2025-06-13  B089ZYJKBK     4.5         2106.0     40615.0       183.0   
2635855 2025-06-13  B0CKYZJY5T     4.2        16413.0      3420.0        16.0   

         salesrank3  bw_pri

In [111]:
df_cleaned.to_csv("0618_amz_pdp_price_sales_gv_top80_data.csv", index=False)